# "가격을 맞혀봐요!" 캡스톤 프로젝트

이번 주 목표 - Amazon 데이터 스크랩을 기반으로 상품 설명에서 가격을 예측하는 모델 만들기

설명만으로 상품 가격을 예측하는 모델입니다.

# 진행 순서

1일차: 데이터 수집 및 정제  
2일차: 데이터 전처리  
3일차: 평가, 기준 모델, 전통적 ML  
4일차: 딥러닝과 LLM  
5일차: 프론티어 모델 파인튜닝  

## 4일차: 신경망과 LLM

오늘은 전통적 ML에서 신경망을 거쳐 대형 언어 모델(LLM)까지 살펴봅니다!!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# 인공 신경망(ANN)을 살펴보기 전에

## 먼저 고려할 수 있는 다른 종류의 신경망이 있습니다

In [ ]:
# 테스트 세트를 CSV로 저장
# human_in.csv를 열어서 2번째 열(0으로 된 부분)에 예상 가격을 직접 입력하세요
# 한국어로 상품을 분석하고 가격을 추정해 입력하면 됩니다

def to_korean_labels(summary):
    """영어 필드명을 한국어로 변환하여 CSV를 더 쉽게 읽을 수 있게 합니다"""
    return (summary
        .replace("Title:", "제목:")
        .replace("Category:", "카테고리:")
        .replace("Brand:", "브랜드:")
        .replace("Description:", "설명:")
        .replace("Details:", "세부사항:"))

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([to_korean_labels(t.summary), 0])

In [ ]:
# 가격을 입력한 CSV 다시 읽기 (human_out.csv)

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"사람이 예측한 가격: ${human} / 실제 가격: ${actual}")


In [ ]:
evaluate(human_pricer, test, size=100)

# 이번엔 - 기본 신경망(vanilla Neural Network)

이 강의의 나머지 부분에서 신경망이 어떻게 작동하는지, 어떻게 학습시키는지 더 깊이 다룰 것입니다.

지금은 맛보기입니다 - PyTorch를 사용하여 처음부터 신경망을 만들어 봅시다.

직관적인 이해를 위한 것이니, 지금 당장 신경망의 모든 것을 알 필요는 없습니다..

In [ ]:
# 문서와 가격 데이터 준비

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# Bag of Words 모델을 위해 HashingVectorizer 사용
# binary=True 설정 시 "원-핫 벡터(one-hot vectors)" 생성

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
# 신경망 정의 - 8개 레이어로 구성된 PyTorch 신경망

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [ ]:
# 데이터를 PyTorch 텐서로 변환
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# 학습 세트와 검증 세트로 분리
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# 데이터 로더 생성
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 모델 초기화
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

In [ ]:
# 손실 함수와 옵티마이저 정의

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 전체 데이터를 2번 순회합니다

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # 다음 4줄은 학습의 4단계입니다: 순전파 → 손실 계산 → 역전파 → 최적화
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'에폭 [{epoch+1}/{EPOCHS}], 학습 손실: {loss.item():.3f}, 검증 손실: {val_loss.item():.3f}')

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
evaluate(neural_network, test)

# 이번엔 - 프론티어 모델로!

추가 학습 없이 순수 추론만으로 프론티어 모델이 얼마나 잘 하는지 확인합니다.

내일은 파인튜닝 학습을 진행합니다.

In [ ]:
def messages_for(item):
    # 한국어로 질문해도 됩니다. 예: "이 제품의 가격을 달러로 예측해주세요."
    message = f"이 제품의 가격을 예측하세요. 가격만 답하고 설명은 하지 마세요.\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [ ]:
# gpt-4.1-nano용 가격 예측 함수

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test)

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)